# 🔬 Computer-Vision Research Template (PyTorch)

A minimal yet **production-ready** notebook that trains ResNet-18 on CIFAR-10, 
logs results to TensorBoard, supports multi-GPU, and can be forked for any new dataset or architecture.

**Author:** Your Name  
**Date:** 2025-07-18  
**Hardware:** GPU with CUDA ≥ 11.8 recommended

In [1]:
# ┌──────────────────────────────────────────────────────────────┐
# │ 0.  Imports & Global Setup                                   │
# └──────────────────────────────────────────────────────────────┘
import os, random, math, json, time, datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## 1️⃣ Hyper-parameters & Config
Centralize every tunable knob so sweeps are trivial (e.g., with Weights&Biases or Hydra).

In [2]:
config = {
    "dataset": "cifar10",
    "data_root": "./data",
    "arch": "resnet18",
    "num_classes": 10,
    "input_size": 32,
    "batch_size": 128,
    "num_epochs": 30,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "momentum": 0.9,
    "num_workers": 4,
    "pin_memory": True,
    "log_every": 50,           # mini-batches
    "log_dir": "runs/cifar10_baseline",
    "save_every": 5,           # epochs
    "checkpoint_dir": "checkpoints"
}

Path(config["log_dir"]).mkdir(parents=True, exist_ok=True)
Path(config["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)

# Pretty-print config
print(json.dumps(config, indent=2))

{
  "dataset": "cifar10",
  "data_root": "./data",
  "arch": "resnet18",
  "num_classes": 10,
  "input_size": 32,
  "batch_size": 128,
  "num_epochs": 30,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "momentum": 0.9,
  "num_workers": 4,
  "pin_memory": true,
  "log_every": 50,
  "log_dir": "runs/cifar10_baseline",
  "save_every": 5,
  "checkpoint_dir": "checkpoints"
}


## 2️⃣ Data Pipeline
- Standard CIFAR-10 augmentation policy (RandomCrop + HorizontalFlip + Normalize).  
- For ImageNet or custom datasets, swap the transforms and dataset class.

In [3]:
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2023, 0.1994, 0.2010)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std)
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std)
])

train_set = torchvision.datasets.CIFAR10(root=config["data_root"], train=True,
                                         download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(root=config["data_root"], train=False,
                                         download=True, transform=test_tf)

train_loader = DataLoader(
    train_set,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=config["num_workers"],
    pin_memory=config["pin_memory"],
    drop_last=True,
)

test_loader = DataLoader(
    test_set,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    pin_memory=config["pin_memory"],
)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:49<00:00, 3.41MB/s]


## 3️⃣ Model & Loss

In [ ]:
def get_model(arch: str, num_classes: int):
    if arch == "resnet18":
        model = resnet18(weights=None)   # random init for CIFAR-10
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(arch)
    return model


model = get_model(config["arch"], config["num_classes"])
model = model.to(DEVICE)

# Wrap for DataParallel / DDP (no-op if single GPU)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(
    model.parameters(),
    lr=config["lr"],
    momentum=config["momentum"],
    weight_decay=config["weight_decay"],
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

## 4️⃣ Utilities: Metric Tracker, Checkpointing

In [ ]:
class AverageMeter:
    """Computes and stores the running average."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def accuracy(output, target, topk=(1,)):
    """Computes top-k accuracy."""
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    return [correct[:k].reshape(-1).float().sum(0) * 100.0 / target.size(0) for k in topk]


def save_checkpoint(state, is_best, path="checkpoint.pth"):
    torch.save(state, path)
    if is_best:
        torch.save(state, path.replace(".pth", "_best.pth"))


writer = SummaryWriter(log_dir=config["log_dir"])
global_step = 0

## 5️⃣ Train & Validate

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


def train_one_epoch(epoch):
    model.train()
    losses = AverageMeter()
    top1 = AverageMeter()
    global global_step

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc1, _ = accuracy(logits, labels, topk=(1, 5))
        losses.update(loss.item(), images.size(0))
        top1.update(acc1.item(), images.size(0))

        if i % config["log_every"] == 0:
            writer.add_scalar("Train/Loss", losses.avg, global_step)
            writer.add_scalar("Train/Acc@1", top1.avg, global_step)
            writer.add_scalar("Train/LR", optimizer.param_groups[0]["lr"], global_step)
            print(
                f"Epoch [{epoch}][{i}/{len(train_loader)}]  "
                f"Loss {losses.val:.3f} ({losses.avg:.3f})  "
                f"Acc@1 {top1.val:.2f} ({top1.avg:.2f})  "
                f"LR {optimizer.param_groups[0]['lr']:.1e}"
            )
        global_step += 1

    scheduler.step(epoch + 1)


@torch.no_grad()
def validate(epoch):
    model.eval()
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    all_logits, all_targets = [], []

    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = criterion(logits, labels)

        acc1, acc5 = accuracy(logits, labels, topk=(1, 5))
        losses.update(loss.item(), images.size(0))
        top1.update(acc1.item(), images.size(0))
        top5.update(acc5.item(), images.size(0))

        all_logits.append(logits.cpu())
        all_targets.append(labels.cpu())

    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)

    writer.add_scalar("Val/Loss", losses.avg, epoch)
    writer.add_scalar("Val/Acc@1", top1.avg, epoch)
    writer.add_scalar("Val/Acc@5", top5.avg, epoch)

    # Add PR curves for TensorBoard
    probs = torch.softmax(all_logits, dim=1)
    writer.add_pr_curve("PR", all_targets, probs[:, 1], epoch)

    print(f"Val [{epoch}]  Loss {losses.avg:.4f}  Acc@1 {top1.avg:.2f}  Acc@5 {top5.avg:.2f}")
    return top1.avg


## 6️⃣ Main Loop

In [ ]:
best_acc = 0.0
start_epoch = 0

# Resume from checkpoint if exists
ckpt_path = Path(config["checkpoint_dir"]) / "latest.pth"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]
    print(f"Resumed from epoch {start_epoch}  best_acc={best_acc:.2f}")

for epoch in range(start_epoch, config["num_epochs"]):
    train_one_epoch(epoch)
    acc = validate(epoch)

    is_best = acc > best_acc
    best_acc = max(acc, best_acc)

    save_checkpoint(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_acc": best_acc,
            "config": config,
        },
        is_best,
        str(ckpt_path),
    )

    if epoch % config["save_every"] == 0:
        torch.save(
            model.state_dict(),
            Path(config["checkpoint_dir"]) / f"epoch_{epoch:03d}.pth"
        )

writer.close()
print("Training complete. Run `tensorboard --logdir runs/` to inspect.")

## 7️⃣ (Optional) Inference on a Single Image

In [ ]:
def imshow(img_tensor, title=""):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")


model.eval()
dataiter = iter(test_loader)
images, labels = next(dataiter)

with torch.no_grad():
    logits = model(images[:4].to(DEVICE))
    preds = logits.argmax(1).cpu()

fig, axs = plt.subplots(1, 4, figsize=(8, 2))
for i, ax in enumerate(axs):
    ax.imshow(np.clip(images[i].numpy().transpose(1, 2, 0) * np.array(std) + np.array(mean), 0, 1))
    ax.set_title(f"GT:{labels[i]}  Pred:{preds[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()